In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import pickle
from pathlib import Path
from scipy import stats
from dataclasses import dataclass, field
from typing import Iterator, Self, Optional
from numpy.typing import ArrayLike
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime as dt, timedelta as td

In [2]:
from argo_interp.data import data_filter, get_data
from argo_interp.cycle.adapter import PchipAdapter
from argo_interp.cycle.model import Model
from argo_interp.cycle.domain import ModelData, ModelMeta
from argo_interp.cycle.config import ModelSettings, ModelKwargs
from argo_interp.model import CycleModels

/home/jcherry/Documents/storage/git/argo-data-interpolation/.venv/bin/python: No module named pip


In [3]:
data_path = Path('./data')
data_path.mkdir(exist_ok=True)

data_file = data_path / 'uncertainty_extension-argo_data.pkl'

In [4]:
override = False

if data_file.exists() and not override:
    with data_file.open('br') as f:
        ds = pickle.load(f)
else:
    box = [
        80, 99, ## Longitude min/max
        6, 23, ## Latitude min/max
        0, 750, ## Pressure min/max
        '2011-01-01', '2020-12-31', ## Datetime min/max
    ]
    ds = get_data(box, progress=True)

    with data_file.open('bw') as f:
        pickle.dump(ds, f)

In [5]:
settings = ModelSettings(n_folds=5)

In [6]:
cycle_models = {}
cycle_data_actuals = {}

cycles = len(ds[['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION']].to_dataframe().drop_duplicates())
t = tqdm(ds.groupby(['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION']), total=cycles)
for (platform_number, cycle_number, direction), cycle_ds in t:
    pressure = cycle_ds['PRES'].values
    temperature = cycle_ds['TEMP'].values
    salinity = cycle_ds['PSAL'].values

    latitude = cycle_ds['LATITUDE'].values[0]
    longitude = cycle_ds['LONGITUDE'].values[0]
    timestamp = cycle_ds['TIME'].values[0]

    if cycle_ds.sizes['N_POINTS'] < 3:
        continue

    cycle_id = f"{int(platform_number)}-{int(cycle_number)}-{direction}"

    model_data = ModelData(
        pressure=pressure,
        temperature=temperature,
        salinity=salinity,
    ).clean_duplicates('mean')

    model_meta = ModelMeta(
        cycle_id=cycle_id,
        latitude=latitude,
        longitude=longitude,
        timestamp=timestamp,
        profile_pressure=(pressure.min(), pressure.max()),
    )

    model = Model.build(model_meta, model_data, PchipAdapter, settings)
    cycle_models[cycle_id] = model
    cycle_data_actuals[cycle_id] = model_data
    t.set_postfix(model_count=len(cycle_models))
cycle_models = CycleModels(models=cycle_models)

  0%|          | 0/21005 [00:00<?, ?it/s]

In [7]:
lat_range = 2
lat_stdev_cutoff = 3

long_range = 2
long_stdev_cutoff = 3

week_range = 8
week_stdev_cutoff = 3

In [8]:
for cycle_id, cycle_meta in cycle_models.index.iterrows():
    pass

In [9]:
lat_filter = [cycle_meta['latitude'] - (lat_range / 2),
              cycle_meta['latitude'] + (lat_range / 2)]

long_filter = [cycle_meta['longitude'] - (long_range / 2),
               cycle_meta['longitude'] + (long_range / 2)]

cycle_filter = [cycle_meta['timestamp'] - td(weeks=week_range / 2),
                cycle_meta['timestamp'] + td(weeks=week_range / 2)]

In [12]:
subcycle_models = cycle_models.filter(lat_filter, long_filter, cyclical_dates=cycle_filter)

In [13]:
subcycle_models

,latitude,longitude,timestamp,pressure_min,pressure_max
cycle_id,,,,,
2900883-238-A,12.531,88.051,2011-02-19 06:54:02,3.6,504.100006
2900883-239-A,12.487,87.946,2011-02-24 07:00:35,4.1,506.000000
2901288-93-A,13.193,87.570,2012-02-07 15:10:53,4.6,750.000000
2901288-94-A,13.053,87.737,2012-02-12 14:43:59,4.8,750.000000
2901288-95-A,12.898,87.629,2012-02-17 15:22:28,5.0,749.599976
...,...,...,...,...,...
6901741-177-A,13.449,87.156,2018-02-10 18:33:00,6.0,738.000000
6901741-178-A,13.359,87.071,2018-02-15 18:39:00,6.0,738.000000
6901741-179-A,13.313,87.007,2018-02-20 18:33:00,6.0,738.000000
